Imports needed for data loading, tokenization, training, and saving.

In [2]:
import random, pickle, heapq

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nltk.tokenize import RegexpTokenizer

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Activation, Embedding
from tensorflow.keras.optimizers import RMSprop

I0000 00:00:1795156545.091675  303819 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795156545.095195  303819 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795156546.129297  303819 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795156547.383708  303819 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

Download the news dataset and print its shape.

In [3]:
# To download and load the csv 
# df = pd.read_csv('https://raw.githubusercontent.com/lutzhamel/fake-news/refs/heads/master/data/fake_or_real_news.csv', index_col=0)

# If locally available
df = pd.read_csv('../data/raw/fake_or_real_news.csv', index_col=0)
print(df.shape)

(6335, 3)


Preview the first rows of the dataset.

In [6]:
df.head()

,title,text,label
id,,,
8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


Collect the article text into one joined string.

In [7]:
text = (df.text.values)
joined_text = " ".join(text)

Keep the first million characters and split them into word tokens.

In [8]:
partial_text = joined_text[:1000000]

tokenizer = RegexpTokenizer(r'\w+')
tokens = tokenizer.tokenize(partial_text.lower())

Build the vocabulary and map each token to an ID.

In [9]:
# Unique Words with ids
unique_tokens = np.unique(tokens)
unique_token_idx = {token: index for index, token in enumerate(unique_tokens)}

Create 10-word input windows with the next word as the target.

In [ ]:
# Building the training dataset
n_words = 10
input_words = [] # 2d array
next_word = [] # 1d array

# For example we have 11 tokens total
# We need to predict the last 1 token
# If we use this formula we can get to the last token
# 11 - 10 = 1 (The one we want to predict)
# Now the loop only runs 0 and 1
for i in range(len(tokens) - n_words):
  input_words.append(tokens[i:i + n_words]) # index 0 [0:0 + 10] -> [0:10]
  next_word.append(tokens[i + n_words]) # index 0 [0 + 10] -> [10]

Convert the word windows into numeric X and y arrays.

In [ ]:
X = np.array([
    [unique_token_idx[word] for word in sequence]
    for sequence in input_words
]) # 2d array

y = np.array([
    unique_token_idx[word]
    for word in next_word
]) # 1d array

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(X[0])

X shape: (169677, 10)
y shape: (169677,)


Define the embedding and LSTM next-word model.

In [ ]:
model = Sequential()
# I Like Machine Learning
# 1 -> 14389
# I -> [1,0,0,0,0,0,0,0,0]
# Like -> [0,1,0,0,0,0,0,0,0]
model.add(Embedding(input_dim=len(unique_tokens), output_dim=128))
model.add(LSTM(128, input_shape=(n_words, len(unique_tokens)), return_sequences=True))
model.add(LSTM(128))
model.add(Dense(len(unique_tokens)))
model.add(Activation('softmax'))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Compile and train the model for 30 epochs.

In [ ]:
# I like machine "learning"
# "learning" - 0.3 || "science" - 0.7 -> loss function
# learning - 0.4 || science - 0.6  -> Loss function
# learning - 0.6 || sceince - 0.4 -> Loss function deccreased
optimizer = RMSprop(learning_rate=0.01)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=['accuracy'])
history = model.fit(X, y, batch_size=128, epochs=30, shuffle=True).history

Epoch 1/30
1326/1326 ━━━━━━━━━━━━━━━━━━━━ 198s 147ms/step - accuracy: 0.0747 - loss: 7.0795
Epoch 2/30
1326/1326 ━━━━━━━━━━━━━━━━━━━━ 201s 146ms/step - accuracy: 0.1110 - loss: 6.6069
Epoch 3/30
1326/1326 ━━━━━━━━━━━━━━━━━━━━ 194s 146ms/step - accuracy: 0.1245 - loss: 6.3996
Epoch 4/30
1326/1326 ━━━━━━━━━━━━━━━━━━━━ 195s 147ms/step - accuracy: 0.1334 - loss: 6.2816
Epoch 5/30
1326/1326 ━━━━━━━━━━━━━━━━━━━━ 201s 146ms/step - accuracy: 0.1385 - loss: 6.1989
Epoch 6/30
1326/1326 ━━━━━━━━━━━━━━━━━━━━ 194s 146ms/step - accuracy: 0.1438 - loss: 6.1378
Epoch 7/30
1326/1326 ━━━━━━━━━━━━━━━━━━━━ 195s 147ms/step - accuracy: 0.1476 - loss: 6.1020
Epoch 8/30
1326/1326 ━━━━━━━━━━━━━━━━━━━━ 194s 146ms/step - accuracy: 0.1502 - loss: 6.0673
Epoch 9/30
1326/1326 ━━━━━━━━━━━━━━━━━━━━ 198s 144ms/step - accuracy: 0.1528 - loss: 6.0427
Epoch 10/30
1326/1326 ━━━━━━━━━━━━━━━━━━━━ 189s 142ms/step - accuracy: 0.1540 - loss: 6.0182
Epoch 11/30
1326/1326 ━━━━━━━━━━━━━━━━━━━━ 190s 143ms/step - accuracy: 0.1563 -

Save the trained model and its training history.

In [ ]:
# Save the model
model.save('../models/keras_next_word_model.keras')

with open("../models/history/history_v1.p", "wb") as f:
    pickle.dump(history, f)

Save the vocabulary and token-to-ID mapping for inference.

In [10]:
with open("../data/processed/unique_tokens.p", "wb") as f:
    pickle.dump(unique_tokens, f)

with open("../data/processed/unique_token_idx.p", "wb") as f:
    pickle.dump(unique_token_idx, f)